# Визуальный baseline: регрессия нутриентов только по фото

Ноутбук решает задачу регрессии четырех целевых переменных — калорий,
жиров, углеводов и белков — только по обзорному снимку блюда, без текста.
Это нужно как baseline: ниже него нельзя; выше должны оказаться
мультимодальные модели в следующих ноутбуках. Архитектура простая и
воспроизводимая: замороженный визуальный энкодер DINOv2-small дает
384-мерный эмбеддинг, поверх которого учится небольшая MLP-голова.
Эмбеддинги кешируются на диск и переиспользуются при перезапуске —
дорогим остается только первый прогон визуального энкодера. Голова
учится за минуты. Точечные предсказания на calibration и test
сохраняются как parquet — на них в ноутбуке 04 строится конформный
интервал.

## 1. Подготовка окружения

Запуск на Kaggle с GPU T4. Внешние зависимости — `transformers` для
загрузки DINOv2-small и `torch` для всего остального; обе уже стоят
в Kaggle-образе. Идемпотентность как в ноутбуке 01: каждый шаг
проверяет наличие итогового артефакта и пропускается, если он есть.

In [1]:
import json
import math
import os
import random
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, TensorDataset
from PIL import Image
from tqdm.auto import tqdm

import matplotlib.pyplot as plt
import seaborn as sns

INPUT_ROOT = Path("/kaggle/input/datasets/dbkarashev/nutrition5k-overhead-rgb-224/n5k_overhead")

WORK_DIR = Path("/kaggle/working/visual_baseline")
EMB_DIR = WORK_DIR / "embeddings"
PRED_DIR = WORK_DIR / "predictions"
EDA_DIR = WORK_DIR / "eda"
WORK_DIR.mkdir(parents=True, exist_ok=True)
for d in (EMB_DIR, PRED_DIR, EDA_DIR):
    d.mkdir(parents=True, exist_ok=True)

CHECKPOINT_FILE = WORK_DIR / "head_checkpoint.pt"
NORM_FILE = WORK_DIR / "target_norm.json"
METRICS_FILE = WORK_DIR / "metrics.json"
STATUS_FILE = WORK_DIR / "_status.json"

ENCODER_NAME = "facebook/dinov2-small"
EMBED_DIM = 384
TARGETS = ["total_calories", "total_fat", "total_carb", "total_protein"]
RANDOM_SEED = 42

ENCODER_BATCH = 64
HEAD_BATCH = 256
HEAD_EPOCHS = 60
HEAD_LR = 1e-3
HEAD_WEIGHT_DECAY = 1e-4
HEAD_HIDDEN = 256
HEAD_DROPOUT = 0.2
EARLY_STOP_PATIENCE = 10


def set_seed(seed: int) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def now_iso() -> str:
    return datetime.now(timezone.utc).isoformat(timespec="seconds").replace("+00:00", "Z")


def update_status(patch: dict) -> None:
    state = {}
    if STATUS_FILE.exists():
        try:
            state = json.loads(STATUS_FILE.read_text())
        except json.JSONDecodeError:
            pass
    state.update(patch)
    state["last_updated"] = now_iso()
    STATUS_FILE.write_text(json.dumps(state, indent=2, ensure_ascii=False))


set_seed(RANDOM_SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"INPUT_ROOT: {INPUT_ROOT}")
print(f"WORK_DIR:   {WORK_DIR}")
print(f"device:     {device}  ({torch.cuda.get_device_name(0) if device.type=='cuda' else 'CPU'})")
print(f"encoder:    {ENCODER_NAME}, embed dim={EMBED_DIM}")
print(f"targets:    {TARGETS}")

INPUT_ROOT: /kaggle/input/datasets/dbkarashev/nutrition5k-overhead-rgb-224/n5k_overhead
WORK_DIR:   /kaggle/working/visual_baseline
device:     cuda  (Tesla T4)
encoder:    facebook/dinov2-small, embed dim=384
targets:    ['total_calories', 'total_fat', 'total_carb', 'total_protein']


Окружение готово. Если на скриншоте `device: cpu`, поменяй в Settings
Accelerator на GPU T4 — без GPU извлечение эмбеддингов будет очень долгим.

## 2. Загрузка метаданных и сплитов

Из подключенного датасета подтягиваем `dishes.parquet` и три текстовых
файла со сплитами. Сравниваем число dish_id со числом фактически
скачанных JPEG в `images_224/` и предупреждаем, если что-то расходится.

In [2]:
META_DIR = INPUT_ROOT / "metadata"
SPLITS_DIR = INPUT_ROOT / "splits"
IMG_DIR = INPUT_ROOT / "images_224"

dishes_df = pd.read_parquet(META_DIR / "dishes.parquet")
ingredients_df = pd.read_parquet(META_DIR / "ingredients.parquet")


def read_split(name: str) -> list[str]:
    return [l.strip() for l in (SPLITS_DIR / f"{name}.txt").read_text().splitlines() if l.strip()]


splits = {name: read_split(name) for name in ("train", "calibration", "test")}
on_disk = {p.stem for p in IMG_DIR.glob("*.jpg")}

for name, ids in splits.items():
    missing = [d for d in ids if d not in on_disk]
    print(f"  {name:<12}{len(ids):4d}  без фото: {len(missing)}")
    if missing:
        splits[name] = [d for d in ids if d in on_disk]

print(f"  блюд в dishes.parquet: {len(dishes_df)}")
print(f"  jpg в images_224:      {len(on_disk)}")

  train       2478  без фото: 0
  calibration  275  без фото: 0
  test         506  без фото: 0
  блюд в dishes.parquet: 5006
  jpg в images_224:      3262


## 3. Нормализация целей

Регрессионную голову учим на нормализованных целях `(y - mean) / std`,
где `mean` и `std` берутся только из train. Это стабилизирует обучение и
делает разные нутриенты сопоставимыми по масштабу. Параметры нормализации
сохраняем в `target_norm.json`, чтобы любой следующий ноутбук мог
восстановить исходные единицы предсказаний.

In [3]:
def get_targets(dish_ids: list[str]) -> tuple[np.ndarray, list[str]]:
    sub = dishes_df.set_index("dish_id").loc[dish_ids]
    return sub[TARGETS].to_numpy(dtype=np.float32), list(sub.index)


if NORM_FILE.exists():
    norm = json.loads(NORM_FILE.read_text())
    print(f"[STEP-3] Уже выполнен, загружаю из {NORM_FILE.name}")
else:
    print("[STEP-3] Запуск")
    y_train, _ = get_targets(splits["train"])
    norm = {
        "targets": TARGETS,
        "mean": y_train.mean(axis=0).tolist(),
        "std": y_train.std(axis=0).tolist(),
    }
    NORM_FILE.write_text(json.dumps(norm, indent=2, ensure_ascii=False))

mean = np.array(norm["mean"], dtype=np.float32)
std = np.array(norm["std"], dtype=np.float32)
for t, m, s in zip(TARGETS, mean, std):
    print(f"  {t:<18} mean={m:8.2f}  std={s:8.2f}")

[STEP-3] Запуск
  total_calories     mean=  252.74  std=  208.13
  total_fat          mean=   12.63  std=   13.30
  total_carb         mean=   19.01  std=   15.73
  total_protein      mean=   18.05  std=   20.08


## 4. Извлечение визуальных эмбеддингов

Для каждого сплита прогоняем изображения через замороженный DINOv2-small
и сохраняем CLS-эмбеддинг как `.npz` (массив + список dish_id). Если файл
для сплита уже есть и в нем столько же строк, сколько dish_id — пересчет
пропускается. В одном проходе через GPU обрабатывается батч по 64 фото.
На T4 все три сплита считаются за пару минут.

In [4]:
class JpgDataset(Dataset):
    def __init__(self, dish_ids: list[str], img_dir: Path, processor):
        self.ids = dish_ids
        self.dir = img_dir
        self.processor = processor

    def __len__(self) -> int:
        return len(self.ids)

    def __getitem__(self, idx: int):
        dish_id = self.ids[idx]
        img = Image.open(self.dir / f"{dish_id}.jpg").convert("RGB")
        pixel = self.processor(images=img, return_tensors="pt")["pixel_values"][0]
        return pixel, dish_id


def split_emb_path(name: str) -> Path:
    return EMB_DIR / f"{name}.npz"


def needs_extraction(name: str) -> bool:
    p = split_emb_path(name)
    if not p.exists():
        return True
    cached = np.load(p, allow_pickle=False)
    return len(cached["dish_ids"]) != len(splits[name])


splits_to_extract = [n for n in ("train", "calibration", "test") if needs_extraction(n)]

if not splits_to_extract:
    print("[STEP-4] Уже выполнен, пропускаю")
else:
    print(f"[STEP-4] Запуск: считаю эмбеддинги для {splits_to_extract}")
    from transformers import AutoImageProcessor, AutoModel
    processor = AutoImageProcessor.from_pretrained(ENCODER_NAME)
    encoder = AutoModel.from_pretrained(ENCODER_NAME).to(device).eval()

    @torch.inference_mode()
    def extract_for(name: str) -> None:
        ids = splits[name]
        ds = JpgDataset(ids, IMG_DIR, processor)
        loader = DataLoader(ds, batch_size=ENCODER_BATCH, shuffle=False, num_workers=2, pin_memory=True)
        embs = np.empty((len(ids), EMBED_DIM), dtype=np.float32)
        all_ids = []
        offset = 0
        for pixels, batch_ids in tqdm(loader, desc=f"{name:<11}"):
            pixels = pixels.to(device, non_blocking=True)
            out = encoder(pixel_values=pixels)
            cls = out.last_hidden_state[:, 0, :]
            n = cls.shape[0]
            embs[offset:offset + n] = cls.float().cpu().numpy()
            all_ids.extend(batch_ids)
            offset += n
        np.savez(split_emb_path(name), embeddings=embs, dish_ids=np.array(all_ids))

    for name in splits_to_extract:
        extract_for(name)

    del encoder
    torch.cuda.empty_cache()


def load_embeddings(name: str) -> tuple[np.ndarray, list[str]]:
    data = np.load(split_emb_path(name), allow_pickle=False)
    return data["embeddings"], list(data["dish_ids"])


for name in ("train", "calibration", "test"):
    embs, ids = load_embeddings(name)
    print(f"  {name:<12}{embs.shape}  ({embs.nbytes / 1024**2:.1f} МБ)")

update_status({"step_4_embeddings_extracted": True})

[STEP-4] Запуск: считаю эмбеддинги для ['train', 'calibration', 'test']


preprocessor_config.json:   0%|          | 0.00/436 [00:00<?, ?B/s]

The image processor of type `BitImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json:   0%|          | 0.00/547 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/88.2M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

train      :   0%|          | 0/39 [00:00<?, ?it/s]

calibration:   0%|          | 0/5 [00:00<?, ?it/s]

test       :   0%|          | 0/8 [00:00<?, ?it/s]

  train       (2478, 384)  (3.6 МБ)
  calibration (275, 384)  (0.4 МБ)
  test        (506, 384)  (0.7 МБ)


Эмбеддинги лежат в `embeddings/{train,calibration,test}.npz`. Размер всех
трех — около пяти мегабайт, попадают в Kaggle Dataset для использования
в следующих ноутбуках без повторного прогона энкодера.

## 5. Обучение регрессионной головы

Голова — двухслойная MLP с дропаутом, выход на четыре нейрона. Лосс —
Smooth L1 на нормализованных целях, оптимизатор AdamW с косинусным
расписанием, ранняя остановка по validation loss на calibration. Лучший
чекпоинт по val loss сохраняется в `head_checkpoint.pt`.

In [5]:
class RegressionHead(nn.Module):
    def __init__(self, in_dim: int, hidden: int, out_dim: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, out_dim),
        )

    def forward(self, x):
        return self.net(x)


def make_loader(name: str, shuffle: bool) -> tuple[DataLoader, list[str]]:
    embs, ids = load_embeddings(name)
    y_raw, ordered_ids = get_targets(ids)
    assert ordered_ids == ids
    y_norm = (y_raw - mean) / std
    ds = TensorDataset(torch.from_numpy(embs), torch.from_numpy(y_norm))
    return DataLoader(ds, batch_size=HEAD_BATCH, shuffle=shuffle, drop_last=False), ids


train_loader, train_ids = make_loader("train", shuffle=True)
val_loader, cal_ids = make_loader("calibration", shuffle=False)

if CHECKPOINT_FILE.exists():
    print("[STEP-5] Уже выполнен, загружаю чекпоинт")
    head = RegressionHead(EMBED_DIM, HEAD_HIDDEN, len(TARGETS), HEAD_DROPOUT).to(device)
    head.load_state_dict(torch.load(CHECKPOINT_FILE, map_location=device))
    history = json.loads((WORK_DIR / "_train_history.json").read_text())
else:
    print("[STEP-5] Запуск")
    head = RegressionHead(EMBED_DIM, HEAD_HIDDEN, len(TARGETS), HEAD_DROPOUT).to(device)
    optimizer = torch.optim.AdamW(head.parameters(), lr=HEAD_LR, weight_decay=HEAD_WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=HEAD_EPOCHS)
    loss_fn = nn.SmoothL1Loss()

    history = {"train_loss": [], "val_loss": []}
    best_val = math.inf
    bad_epochs = 0
    best_state = None

    for epoch in range(1, HEAD_EPOCHS + 1):
        head.train()
        train_losses = []
        for xb, yb in train_loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            pred = head(xb)
            loss = loss_fn(pred, yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())
        scheduler.step()

        head.eval()
        with torch.inference_mode():
            val_losses = []
            for xb, yb in val_loader:
                xb = xb.to(device, non_blocking=True)
                yb = yb.to(device, non_blocking=True)
                val_losses.append(loss_fn(head(xb), yb).item())

        train_loss = float(np.mean(train_losses))
        val_loss = float(np.mean(val_losses))
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        improved = val_loss < best_val - 1e-5
        if improved:
            best_val = val_loss
            bad_epochs = 0
            best_state = {k: v.detach().cpu().clone() for k, v in head.state_dict().items()}
        else:
            bad_epochs += 1

        marker = "  *" if improved else ""
        print(f"  epoch {epoch:3d}  train={train_loss:.4f}  val={val_loss:.4f}{marker}")

        if bad_epochs >= EARLY_STOP_PATIENCE:
            print(f"  Ранняя остановка: {EARLY_STOP_PATIENCE} эпох без улучшения val")
            break

    head.load_state_dict(best_state)
    torch.save(best_state, CHECKPOINT_FILE)
    (WORK_DIR / "_train_history.json").write_text(json.dumps(history, indent=2))

print(f"  лучший val Smooth L1 (норм.): {min(history['val_loss']):.4f}")
update_status({"step_5_head_trained": True})

[STEP-5] Запуск
  epoch   1  train=0.2575  val=0.2495  *
  epoch   2  train=0.1730  val=0.1918  *
  epoch   3  train=0.1402  val=0.1724  *
  epoch   4  train=0.1210  val=0.1580  *
  epoch   5  train=0.1077  val=0.1497  *
  epoch   6  train=0.0975  val=0.1399  *
  epoch   7  train=0.0893  val=0.1377  *
  epoch   8  train=0.0845  val=0.1220  *
  epoch   9  train=0.0748  val=0.1259
  epoch  10  train=0.0701  val=0.1130  *
  epoch  11  train=0.0631  val=0.1086  *
  epoch  12  train=0.0629  val=0.1038  *
  epoch  13  train=0.0571  val=0.1057
  epoch  14  train=0.0514  val=0.1024  *
  epoch  15  train=0.0507  val=0.0973  *
  epoch  16  train=0.0487  val=0.1070
  epoch  17  train=0.0449  val=0.1012
  epoch  18  train=0.0427  val=0.1125
  epoch  19  train=0.0422  val=0.1097
  epoch  20  train=0.0379  val=0.0976
  epoch  21  train=0.0381  val=0.1030
  epoch  22  train=0.0355  val=0.1031
  epoch  23  train=0.0348  val=0.1005
  epoch  24  train=0.0354  val=0.1041
  epoch  25  train=0.0329  val=0.

Голова обучена и сохранена. История лоссов лежит в `_train_history.json`.

## 6. Оценка на test

Для каждой цели считаем MAE и MAPE в исходных единицах, плюс R^2.
Точечные предсказания сохраняем для test и calibration в parquet.
Calibration-предсказания нужны для ноутбука 04 (конформная калибровка)
— поэтому сохраняем их как самостоятельный артефакт.

In [6]:
@torch.inference_mode()
def predict(name: str) -> tuple[pd.DataFrame, np.ndarray, np.ndarray]:
    embs, ids = load_embeddings(name)
    y_true, _ = get_targets(ids)
    head.eval()
    xb = torch.from_numpy(embs).to(device)
    pred_norm = head(xb).cpu().numpy()
    pred = pred_norm * std + mean

    df = pd.DataFrame({"dish_id": ids})
    for i, t in enumerate(TARGETS):
        df[f"pred_{t}"] = pred[:, i]
        df[f"true_{t}"] = y_true[:, i]
    return df, y_true, pred


def per_target_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    out = {}
    eps = 1e-6
    for i, t in enumerate(TARGETS):
        yt = y_true[:, i]
        yp = y_pred[:, i]
        mae = float(np.mean(np.abs(yp - yt)))
        mape = float(np.mean(np.abs(yp - yt) / np.maximum(np.abs(yt), eps)) * 100)
        ss_res = float(np.sum((yp - yt) ** 2))
        ss_tot = float(np.sum((yt - yt.mean()) ** 2))
        r2 = 1.0 - ss_res / max(ss_tot, eps)
        out[t] = {"mae": mae, "mape_percent": mape, "r2": r2}
    return out


df_cal, y_cal, p_cal = predict("calibration")
df_test, y_test, p_test = predict("test")
df_cal.to_parquet(PRED_DIR / "calibration.parquet", index=False)
df_test.to_parquet(PRED_DIR / "test.parquet", index=False)

metrics = {
    "calibration": per_target_metrics(y_cal, p_cal),
    "test": per_target_metrics(y_test, p_test),
}
METRICS_FILE.write_text(json.dumps(metrics, indent=2, ensure_ascii=False))

print("Метрики на test:")
print(f"  {'target':<18}{'MAE':>10}{'MAPE %':>10}{'R^2':>10}")
for t in TARGETS:
    m = metrics["test"][t]
    print(f"  {t:<18}{m['mae']:10.2f}{m['mape_percent']:10.1f}{m['r2']:10.3f}")

update_status({"step_6_evaluated": True})

Метрики на test:
  target                   MAE    MAPE %       R^2
  total_calories         64.15      41.5     0.783
  total_fat               4.63  587256.4     0.724
  total_carb              6.14 3739925.0     0.724
  total_protein           6.05      95.2     0.740


## 7. Анализ ошибок

Три графика. `training_curves.png` — кривые train/val Smooth L1.
`residuals.png` — четыре scatter (predicted vs true) для каждой цели на
test, с диагональю идеального предсказания. `per_target_metrics.png` —
бар-чарт MAE и MAPE по целям.

In [7]:
sns.set_theme(style="whitegrid", context="talk")

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(history["train_loss"], label="train", color="steelblue")
ax.plot(history["val_loss"], label="val (calibration)", color="darkorange")
ax.set(xlabel="Эпоха", ylabel="Smooth L1 (норм.)", title="Кривые обучения головы")
ax.legend()
fig.tight_layout()
fig.savefig(EDA_DIR / "training_curves.png", dpi=120)
plt.close(fig)

fig, axes = plt.subplots(2, 2, figsize=(13, 11))
units = {"total_calories": "ккал", "total_fat": "г", "total_carb": "г", "total_protein": "г"}
for ax, (i, t) in zip(axes.flat, enumerate(TARGETS)):
    yt = y_test[:, i]
    yp = p_test[:, i]
    ax.scatter(yt, yp, s=8, alpha=0.4, color="steelblue")
    lo, hi = float(min(yt.min(), yp.min())), float(max(yt.max(), yp.max()))
    ax.plot([lo, hi], [lo, hi], "r--", lw=1)
    ax.set(xlabel=f"истина, {units[t]}", ylabel=f"предсказание, {units[t]}", title=t)
fig.suptitle("Test: предсказание против истины", fontsize=16)
fig.tight_layout()
fig.savefig(EDA_DIR / "residuals.png", dpi=120)
plt.close(fig)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
mae_vals = [metrics["test"][t]["mae"] for t in TARGETS]
mape_vals = [metrics["test"][t]["mape_percent"] for t in TARGETS]
axes[0].bar(TARGETS, mae_vals, color="steelblue")
axes[0].set(title="MAE на test", ylabel="MAE (исходные единицы)")
axes[0].tick_params(axis="x", rotation=20)
axes[1].bar(TARGETS, mape_vals, color="darkorange")
axes[1].set(title="MAPE на test", ylabel="MAPE, %")
axes[1].tick_params(axis="x", rotation=20)
fig.tight_layout()
fig.savefig(EDA_DIR / "per_target_metrics.png", dpi=120)
plt.close(fig)

print(f"  графики в {EDA_DIR}: {sorted(p.name for p in EDA_DIR.glob('*.png'))}")
update_status({"step_7_eda_completed": True})

  графики в /kaggle/working/visual_baseline/eda: ['per_target_metrics.png', 'residuals.png', 'training_curves.png']


## 8. Финальный лог состояния

Сохраняем сводку: метрики на test, размеры артефактов, конфиг
обучения. Этот ноутбук готов к Save Version. После него содержимое
`/kaggle/working/visual_baseline/` стоит опубликовать как датасет
`nutrition5k-visual-baseline` — он будет input для ноутбука 04.

In [8]:
def dir_size_mb(p: Path) -> float:
    return sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) / (1024 * 1024)


summary = {
    "encoder": ENCODER_NAME,
    "embed_dim": EMBED_DIM,
    "targets": TARGETS,
    "head": {
        "hidden": HEAD_HIDDEN,
        "dropout": HEAD_DROPOUT,
        "epochs_run": len(history["train_loss"]),
        "best_val_smoothl1_norm": float(min(history["val_loss"])),
    },
    "test_metrics": metrics["test"],
    "calibration_metrics": metrics["calibration"],
    "counts": {
        "train": len(splits["train"]),
        "calibration": len(splits["calibration"]),
        "test": len(splits["test"]),
    },
    "sizes_mb": {
        "embeddings": dir_size_mb(EMB_DIR),
        "predictions": dir_size_mb(PRED_DIR),
        "eda": dir_size_mb(EDA_DIR),
        "checkpoint": CHECKPOINT_FILE.stat().st_size / 1024**2,
    },
    "config": {
        "head_lr": HEAD_LR,
        "head_weight_decay": HEAD_WEIGHT_DECAY,
        "head_batch": HEAD_BATCH,
        "encoder_batch": ENCODER_BATCH,
        "early_stop_patience": EARLY_STOP_PATIENCE,
        "random_seed": RANDOM_SEED,
    },
    "timestamp": now_iso(),
}
update_status({
    "step_8_summary_written": True,
    "summary": summary,
})

print(json.dumps(summary, indent=2, ensure_ascii=False))
print()
print("Готово. Сделай Save & Run All (Commit) и опубликуй output как")
print("приватный датасет nutrition5k-visual-baseline для ноутбука 04.")

{
  "encoder": "facebook/dinov2-small",
  "embed_dim": 384,
  "targets": [
    "total_calories",
    "total_fat",
    "total_carb",
    "total_protein"
  ],
  "head": {
    "hidden": 256,
    "dropout": 0.2,
    "epochs_run": 36,
    "best_val_smoothl1_norm": 0.09514627978205681
  },
  "test_metrics": {
    "total_calories": {
      "mae": 64.14520263671875,
      "mape_percent": 41.4918327331543,
      "r2": 0.7829377226039075
    },
    "total_fat": {
      "mae": 4.631979942321777,
      "mape_percent": 587256.375,
      "r2": 0.7237185405008655
    },
    "total_carb": {
      "mae": 6.144895553588867,
      "mape_percent": 3739925.0,
      "r2": 0.7240757472177446
    },
    "total_protein": {
      "mae": 6.050958633422852,
      "mape_percent": 95.20071411132812,
      "r2": 0.7397716422327807
    }
  },
  "calibration_metrics": {
    "total_calories": {
      "mae": 54.65715026855469,
      "mape_percent": 45.02077102661133,
      "r2": 0.8570639795963017
    },
    "total_fat"